# 01 · Data Ingestion & Chunking

**Goal:** turn raw bank filings (PDFs) into small, clean, labelled pieces of text ("chunks") for embedding and searching.

In [ ]:
%pip install -q pymupdf beautifulsoup4 lxml pandas matplotlib tiktoken tqdm requests

## 1. Setup

In [ ]:
import itertools, json, re, runpy, unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pymupdf as fitz
import tiktoken
from bs4 import BeautifulSoup
from IPython.display import display
from tqdm.auto import tqdm

ROOT = Path.cwd()
BASE = ROOT.parent if ROOT.name == 'notebooks' else ROOT
RAW = BASE / 'data' / 'raw'                       # downloaded filings
PROCESSED = BASE / 'data' / 'processed'           # everything this notebook produces
TEXT_DIR = PROCESSED / 'text'                     # cleaned text, one file per filing
MANIFEST = BASE / 'data' / 'filing_manifest.csv'  # written by the download script
for folder in (RAW, TEXT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

BANKS = {'RBC': 'Royal Bank of Canada',
         'TD': 'Toronto-Dominion Bank',
         'BMO': 'Bank of Montreal',
         'BNS': 'Bank of Nova Scotia',
         'CIBC': 'Canadian Imperial Bank of Commerce'}
TARGET_YEARS = [2024, 2025]   # last two completed fiscal years (the banks' years end on October 31)

# Chunk sizes in tokens (from the module: 500-1000 per chunk, 100-200 overlap)
CHUNK_MIN, CHUNK_TARGET, CHUNK_MAX = 500, 750, 1000
OVERLAP_MIN, OVERLAP_MAX = 150, 200
EMBEDDING_USD_PER_MILLION = 0.02   # OpenAI text-embedding-3-small

enc = tiktoken.get_encoding('cl100k_base')   # downloads its vocabulary on first use (needs internet)

def count_tokens(text):
    return len(enc.encode(text, disallowed_special=()))

## 2. Download the filings

The script `demos/01_download_filings.py` saves each filing into `data/raw/` (files already there are skipped).

In [ ]:
downloader = runpy.run_path(str(BASE / 'demos' / '01_download_filings.py'))
report = downloader['download_filings'](base=BASE, years=TARGET_YEARS)
for error in report['errors']:
    print('Download problem:', error)

files = sorted(p for p in RAW.rglob('*') if p.suffix.lower() in {'.pdf', '.html', '.htm'})
print(f'{len(files)} filings in {RAW}')

## 3. Work out what each file is

Turn filename into **metadata** that travels with every chunk, so later we can cite sources and filter searches ("only RBC, only 2025").

In [ ]:
# Read source links and filing dates when the downloader provides them.
source_urls, filing_dates = {}, {}
if MANIFEST.exists():
    manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
    if {'file_name', 'source_url'}.issubset(manifest.columns):
        source_urls = dict(zip(manifest['file_name'], manifest['source_url']))
    if {'file_name', 'filing_date'}.issubset(manifest.columns):
        filing_dates = dict(zip(manifest['file_name'], manifest['filing_date']))

def filing_metadata(path):
    """Read bank, year, document type and quarter from the filename."""
    parts = path.stem.upper().split('_')
    code, year = parts[0], int(parts[1])
    quarter = next((int(part[1]) for part in parts[2:] if re.fullmatch(r'Q[1-4]', part)), None)
    if quarter:
        doc_type = 'quarterly'
    elif 'AIF' in parts:
        doc_type = 'AIF'
    else:
        doc_type = 'annual_report'
    return {
        'filing_id': f"{code}_{year}_{doc_type}_{'Q' + str(quarter) if quarter else 'FY'}",
        'bank': BANKS[code], 'bank_code': code, 'document_type': doc_type,
        'fiscal_year': year, 'quarter': quarter, 'file_name': path.name,
        'source_url': source_urls.get(path.name) or None,
        # Never treat the fiscal year as the actual publication date.
        'filing_date': filing_dates.get(path.name) or None,
    }

pd.DataFrame([filing_metadata(p) for p in files]).head()

## 4. Read the pages

Each page is kept as `(page_number, text)` so chunks can cite their page range later.

In [ ]:
def read_pages(path):
    """Return a list of (page_number, text). HTML has no pages, so it comes back as one item."""
    if path.suffix.lower() == '.pdf':
        with fitz.open(path) as pdf:
            return [(i + 1, '\n\n'.join(b[4].strip() for b in page.get_text('blocks', sort=True) if b[6] == 0))
                    for i, page in enumerate(pdf)]

    soup = BeautifulSoup(path.read_bytes(), 'lxml')
    for tag in soup(['script', 'style', 'nav', 'header', 'footer']):
        tag.decompose()
    for table in soup.find_all('table'):   # keep each table together as rows of 'a | b | c'
        rows = [' | '.join(c.get_text(' ', strip=True) for c in tr.find_all(['th', 'td'])) for tr in table.find_all('tr')]
        table.replace_with('\n\n' + '\n'.join(rows) + '\n\n')
    for tag in soup.find_all(['p', 'div', 'h1', 'h2', 'h3', 'h4', 'li', 'br']):
        tag.insert_before('\n\n')
    return [(None, soup.get_text())]

pages = read_pages(files[0])
print(files[0].name, '|', len(pages), 'pages')
print(pages[0][1][:300])

## 5. Clean the text

Removed here:

- **Headers, footers and page numbers**: lines that repeat at the top or bottom of most pages
- **Table-of-contents pages**
- **French pages**: many filings are bilingual and we only want English

In [ ]:
def normalize_text(text):
    """Tidy odd characters (e.g. the 'ﬁ' ligature becomes 'fi') and extra spaces."""
    text = unicodedata.normalize('NFKC', text).replace('\xa0', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    return re.sub(r' *\n *', '\n', text).strip()

def strip_page_number(line):
    """'Risk Management 47' -> 'Risk Management', so a running header matches on every page."""
    return re.sub(r'\s+\d{1,4}$', '', line.strip()).strip(' -|')

def edge_lines(text):
    """First two and last two non-empty lines of a page, where headers and footers live."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[:2] + lines[-2:]

def find_repeated_headers(pages, min_share=0.6):
    """Header/footer lines that appear on at least 60% of pages (lines with $ or % are real figures, so kept)."""
    if len(pages) < 5:
        return set()
    counts = Counter(l for _, text in pages for l in {strip_page_number(x) for x in edge_lines(text)})
    return {l for l, n in counts.items()
            if n / len(pages) >= min_share and 5 <= len(l) <= 120 and not re.search(r'[$%]', l)}

def is_contents_page(lines):
    dotted_rows = sum(bool(re.search(r'\.{3,}\s*\d+$', l)) for l in lines)   # "Credit risk ........ 45"
    return dotted_rows >= 5 or any(re.fullmatch(r'(?:table of )?contents', l, re.I) for l in lines[:5])

def is_french_page(text):
    french = len(re.findall(r'\b(?:les|des|dans|nous|notre|pour|avec|une)\b', text, re.I))
    english = len(re.findall(r'\b(?:the|and|with|our|for|from|was|were)\b', text, re.I))
    return french > 5 and french > 2 * english

def clean_pages(pages):
    """Drop contents/French pages, then remove headers, footers and page numbers from the rest."""
    headers = find_repeated_headers(pages)
    cleaned = []
    for number, text in pages:
        lines = text.splitlines()
        if is_contents_page([l.strip() for l in lines if l.strip()]) or is_french_page(text):
            continue
        non_empty = [i for i, l in enumerate(lines) if l.strip()]
        edges = set(non_empty[:2] + non_empty[-2:])
        kept = [l for i, l in enumerate(lines)
                if not (i in edges and (strip_page_number(l) in headers
                                        or re.fullmatch(r'(?:page\s*)?\d{1,4}', l.strip(), re.I)))]
        cleaned.append((number, normalize_text('\n'.join(kept))))
    return cleaned

cleaned = clean_pages(pages)
print(f'{len(pages)} pages -> {len(cleaned)} pages kept')

## 6. Split into paragraphs and track sections

We walk through the text line by line:

- A short line that looks like a **heading** (a known name like "Credit Risk", or ALL CAPS) starts a new section.
- Blank lines separate **paragraphs**.
- **Forward-looking statement** sections are legal boilerplate that appears in every filing and adds noise, so we skip them.
- A block with several lines full of numbers is treated as a **table** and later kept in one piece.

In [ ]:
HEADING = re.compile(
    r"(?:management['’]s discussion and analysis|md&a|risk management|credit risk|market risk|operational risk|"
    r"liquidity risk|climate(?:-related)? risk|financial statements|capital management|risk factors|"
    r"corporate governance|business overview|overview|"
    r"notes to (?:the )?(?:consolidated )?financial statements|"
    r"forward[- ]looking statements?|caution(?:ary)? (?:note|statement).*)", re.I)
BOILERPLATE = re.compile(r'forward[- ]looking|caution(?:ary)? (?:note|statement)', re.I)

def is_heading(line):
    line = line.strip().strip(':')
    if not 3 <= len(line) <= 110 or len(line.split()) > 13:
        return False
    if HEADING.fullmatch(line):
        return True
    letters = ''.join(c for c in line if c.isalpha())   # ALL-CAPS line without numbers or symbols
    return len(letters) >= 5 and letters.isupper() and not re.search(r'[$%\d|]', line)

def looks_like_table(lines):
    return sum(len(re.findall(r'\d[\d,.]*', l)) >= 3 for l in lines) >= 2

def to_paragraphs(pages):
    """Return paragraphs as dicts: text, section, page, is_table."""
    paragraphs, section, skipping = [], 'Unclassified', False
    for page_number, text in pages:
        block = []
        for line in text.splitlines() + ['']:        # the extra '' ends the last paragraph on the page
            line = line.strip()
            if line and not is_heading(line):
                block.append(line)
                continue
            if block and not skipping:
                paragraphs.append({'text': '\n'.join(block), 'section': section,
                                   'page': page_number, 'is_table': looks_like_table(block)})
            block = []
            if line:                                  # this line was a heading
                section = line.strip(':')
                skipping = bool(BOILERPLATE.search(section))
    return paragraphs

paragraphs = to_paragraphs(cleaned)
print(len(paragraphs), 'paragraphs; first sections:', list(dict.fromkeys(p['section'] for p in paragraphs))[:6])

## 7. Chunk at paragraph boundaries

- Aim for **500–1,000 tokens**, with up to **200 tokens of overlap**.
- Keep sentences together when possible, and keep short paragraphs in the same section together.
- Keep tables intact if they fit. For long tables, split at row boundaries so no chunk exceeds the limit.
- Very short chunks may remain at the end of a section; don't mix unrelated sections just to reach 500 tokens.


In [ ]:
def split_sentences(text):
    """Split at sentence endings where possible."""
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z(])', text) if s.strip()]

def split_to_fit(text, max_tokens=CHUNK_MAX):
    """Split very long text by tokens if it has no usable sentence breaks."""
    tokens = enc.encode(text, disallowed_special=())
    return [enc.decode(tokens[i:i + max_tokens]).strip()
            for i in range(0, len(tokens), max_tokens)]

def split_long_paragraph(text, is_table=False):
    """Keep sentences or table rows together unless one is too long."""
    if count_tokens(text) <= CHUNK_MAX:
        return [text]

    parts = text.splitlines() if is_table else split_sentences(text)
    pieces, current = [], ''
    for part in parts:
        # A single table row or sentence may itself be longer than 1,000 tokens.
        for item in split_to_fit(part) if count_tokens(part) > CHUNK_MAX else [part]:
            separator = '\n' if is_table else ' '
            candidate = (current + separator + item).strip() if current else item
            if current and count_tokens(candidate) > CHUNK_MAX:
                pieces.append(current)
                current = item
            else:
                current = candidate
    if current:
        pieces.append(current)
    return pieces

def overlap_text(units):
    """Take up to 200 tokens from the end of the previous chunk, excluding tables."""
    sentences = []
    for unit in units:
        sentences = [] if unit['is_table'] else sentences + split_sentences(unit['text'])

    picked, total = [], 0
    for sentence in reversed(sentences):
        tokens = count_tokens(sentence)
        if total + tokens > OVERLAP_MAX:
            break
        picked.insert(0, sentence)
        total += tokens
        if total >= OVERLAP_MIN:
            break
    return ' '.join(picked)

def build_chunk(units, section, meta, number):
    text = '\n\n'.join(unit['text'] for unit in units)
    pages = [unit['page'] for unit in units if unit['page'] is not None]
    return {
        **meta, 'chunk_id': f"{meta['filing_id']}_{number:05d}",
        'section': section, 'text': text,
        'page_start': min(pages) if pages else None,
        'page_end': max(pages) if pages else None,
        'token_count': count_tokens(text),
        'has_table': any(unit['is_table'] for unit in units),
    }

def make_chunks(paragraphs, meta):
    chunks = []
    for section, group in itertools.groupby(paragraphs, key=lambda p: p['section']):
        units = []
        for paragraph in group:
            for piece in split_long_paragraph(paragraph['text'], paragraph['is_table']):
                if piece.strip():
                    units.append({**paragraph, 'text': piece,
                                  'tokens': count_tokens(piece)})

        current = []
        for unit in units:
            combined = '\n\n'.join([u['text'] for u in current] + [unit['text']])
            if current and (count_tokens(combined) > CHUNK_MAX or
                            (count_tokens(combined) > CHUNK_TARGET and
                             count_tokens('\n\n'.join(u['text'] for u in current)) >= CHUNK_MIN)):
                chunks.append(build_chunk(current, section, meta, len(chunks)))
                # Add overlap only when the next piece still fits.
                overlap = overlap_text(current)
                current = []
                if overlap and count_tokens(overlap + '\n\n' + unit['text']) <= CHUNK_MAX:
                    current.append({'text': overlap, 'page': chunks[-1]['page_end'],
                                    'is_table': False, 'tokens': count_tokens(overlap)})
            current.append(unit)

        if current:
            # Avoid a tiny final chunk when it fits in the preceding chunk.
            final = build_chunk(current, section, meta, len(chunks))
            if (final['token_count'] < CHUNK_MIN and chunks and
                    chunks[-1]['filing_id'] == meta['filing_id'] and
                    chunks[-1]['section'] == section):
                previous = chunks[-1]
                combined = previous['text'] + '\n\n' + final['text']
                if count_tokens(combined) <= CHUNK_MAX:
                    previous['text'] = combined
                    previous['token_count'] = count_tokens(combined)
                    previous['page_end'] = final['page_end']
                    previous['has_table'] |= final['has_table']
                    continue
            chunks.append(final)

    assert all(c['token_count'] <= CHUNK_MAX for c in chunks), 'Chunk exceeds token limit'
    return chunks

sample = make_chunks(paragraphs, filing_metadata(files[0]))
print(len(sample), 'chunks; first token counts:', [c['token_count'] for c in sample[:8]])


## 8. Run on every filing and save

Outputs in `data/processed/`:

| File | Contents |
|---|---|
| `filing_index.csv` | one row per filing: bank, type, year, quarter, date, pages, chunks, tokens |
| `chunks.jsonl` | every chunk with its text and metadata (input for Step 2) |
| `text/*.txt` | cleaned text of each filing |

In [ ]:
chunks, index_rows, errors = [], [], []

for path in tqdm(files, desc='Processing filings'):
    try:
        meta = filing_metadata(path)
        raw_pages = read_pages(path)
        paragraphs = to_paragraphs(clean_pages(raw_pages))
        filing_chunks = make_chunks(paragraphs, meta)
        if not filing_chunks:
            raise ValueError('no usable text (a scanned PDF would need OCR)')
    except Exception as exc:
        errors.append({'file': path.name, 'error': str(exc)})
        continue

    text = '\n\n'.join(f"{p['section']}\n{p['text']}" for p in paragraphs)
    (TEXT_DIR / f"{meta['filing_id']}.txt").write_text(text, encoding='utf-8')
    chunks += filing_chunks
    index_rows.append({**meta, 'pages': len(raw_pages), 'chunks': len(filing_chunks),
                       'tokens': sum(c['token_count'] for c in filing_chunks)})

assert chunks, 'No chunks were created. Check data/raw and the errors above.'

index_df, chunks_df = pd.DataFrame(index_rows), pd.DataFrame(chunks)
index_df.to_csv(PROCESSED / 'filing_index.csv', index=False)
with open(PROCESSED / 'chunks.jsonl', 'w', encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f'{len(index_df)} filings -> {len(chunks):,} chunks ({len(errors)} errors)')
if errors:
    display(pd.DataFrame(errors))
display(index_df[['bank_code', 'document_type', 'fiscal_year', 'quarter', 'filing_date', 'pages', 'chunks']])

## 9. Token counts and embedding cost

In [ ]:
total_tokens = int(chunks_df['token_count'].sum())
print(f'Chunks: {len(chunks_df):,}')
print(f'Total tokens (including overlap): {total_tokens:,}')
print(f'Estimated embedding cost: ${total_tokens / 1e6 * EMBEDDING_USD_PER_MILLION:.2f} '
      f'(OpenAI text-embedding-3-small at ${EMBEDDING_USD_PER_MILLION} per million tokens)')
display(chunks_df['token_count'].describe().round(0).to_frame('tokens per chunk'))

short = (chunks_df['token_count'] < CHUNK_MIN).mean()
print(f'\n{short:.0%} of chunks are shorter than {CHUNK_MIN} tokens. These are usually tiny sections or the last chunk of a section,')
print('because chunks never cross a section boundary. Very short chunks carry little meaning.')

## 10. Charts: chunk lengths and chunks per bank

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(chunks_df['token_count'], bins=35, edgecolor='white')
ax.axvline(CHUNK_MIN, linestyle='--', color='gray', label=f'target min ({CHUNK_MIN})')
ax.axvline(CHUNK_MAX, linestyle='--', color='black', label=f'max ({CHUNK_MAX})')
ax.set(title='Chunk length distribution', xlabel='Tokens per chunk', ylabel='Number of chunks')
ax.legend()
fig.tight_layout()
fig.savefig(PROCESSED / 'chunk_length_distribution.png', dpi=150)
plt.show()

counts = chunks_df.groupby(['bank_code', 'document_type']).size().unstack(fill_value=0)
ax = counts.plot(kind='bar', figsize=(10, 4), rot=0)
ax.set(title='Chunks per bank and document type', xlabel='Bank', ylabel='Number of chunks')
plt.tight_layout()
plt.savefig(PROCESSED / 'chunks_by_bank_and_type.png', dpi=150)
plt.show()

## 11. Check data coverage and chunk quality before Step 2

Look for missing bank filings and unusually short chunks. A blank filing date means the downloader did not supply a verified date; it should not be filled with a guessed date.


In [ ]:
display(index_df.pivot_table(index='bank_code', columns=['fiscal_year', 'document_type'], values='chunks', aggfunc='count', fill_value=0))

# Annual reports and AIFs are expected for each bank/year; quarterly reports should
# usually cover Q1–Q3 (Q4 information is included in the annual report).
missing = []
for bank in BANKS:
    for year in TARGET_YEARS:
        found = index_df[(index_df.bank_code == bank) & (index_df.fiscal_year == year)]
        for doc_type in ('annual_report', 'AIF'):
            if not (found.document_type == doc_type).any():
                missing.append(f'{bank} {year}: missing {doc_type}')
        quarters = set(found.loc[found.document_type == 'quarterly', 'quarter'].dropna().astype(int))
        for quarter in (1, 2, 3):
            if quarter not in quarters:
                missing.append(f'{bank} {year}: missing Q{quarter} quarterly report')

print('Missing expected filings:')
print('\n'.join(missing) if missing else 'None found')
print('Missing verified filing dates:', index_df['filing_date'].isna().sum())
print('Chunks exceeding 1,000 tokens:', int((chunks_df.token_count > CHUNK_MAX).sum()))
print('Chunks below 500 tokens:', int((chunks_df.token_count < CHUNK_MIN).sum()))
print('Chunks with no section:', int((chunks_df.section == 'Unclassified').sum()))
print('\nSample chunk:')
print(chunks_df.iloc[len(chunks_df) // 2][['chunk_id', 'section', 'page_start', 'page_end']].to_string())
print(chunks_df.iloc[len(chunks_df) // 2]['text'][:800])
